# Neural Topology Mapping — Crowd Surge & Compression Mitigation (Phase 2)
### Google Colab GPU Deployment with Cloudflare Tunnel

> **Important Note on Colab Session Limits:**
- Free Google Colab instances allow continuous execution for up to 12 hours (or earlier if idle for ~30-60 minutes).
- Keep this browser tab open to prevent Colab from idling and killing the background server.
- All uploads and database entries in the Colab disk are ephemeral and reset when the runtime terminates.

### Step 1: Verify Hardware Accelerator (NVIDIA CUDA GPU)

In [ ]:
!nvidia-smi

# Clone the repository (or pull latest updates)
!git clone https://github.com/rrohancodes/mini_project.git || (cd mini_project && git reset --hard && git pull)
%cd mini_project

# Install FastAPI, imageio-ffmpeg, and ultralytics for GPU YOLO inference
!pip install -q -r requirements.txt ultralytics lapx


In [ ]:
# Clone the repository (or pull latest updates)
!git clone https://github.com/rrohancodes/mini_project.git || (cd mini_project && git pull)
%cd mini_project

# Install FastAPI, imageio-ffmpeg, and ultralytics for GPU YOLO inference
!pip install -q -r requirements.txt ultralytics lapx


import subprocess
import time
import requests

# 1. Kill any previous server processes holding port 8000
!fuser -k 8000/tcp 2>/dev/null || pkill -f 'server.py' 2>/dev/null
time.sleep(1.5)

# 2. Launch server.py in background
server_log = open('server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', 'server.py', '--port', '8000'],
    stdout=server_log,
    stderr=subprocess.STDOUT
)

# 3. Wait until server responds to health check
print('Waiting for FastAPI server to initialize on port 8000...')
ready = False
for _ in range(30):
    try:
        res = requests.get('http://localhost:8000/api/info', timeout=1)
        if res.status_code == 200:
            info = res.json()
            print(f'Server is ONLINE!')
            print(f'Model: {info.get("model")}')
            print(f'Device: {info.get("device")}')
            ready = True
            break
    except Exception:
        time.sleep(1)

if not ready:
    print('Warning: Server did not respond within 30 seconds. Check server.log below:')
    !cat server.log


In [ ]:
import subprocess
import time
import requests

# Launch server.py in background
server_log = open('server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', 'server.py', '--port', '8000'],
    stdout=server_log,
    stderr=subprocess.STDOUT
)

# Wait until server responds to health check
print('Waiting for FastAPI server to initialize on port 8000...')
ready = False
for _ in range(30):
    try:
        res = requests.get('http://localhost:8000/api/info', timeout=1)
        if res.status_code == 200:
            info = res.json()
            print(f'Server is ONLINE!')
            print(f'Model: {info.get("model")}')
            print(f'Device: {info.get("device")}')
            ready = True
            break
    except Exception:
        time.sleep(1)

if not ready:
    print('Warning: Server did not respond within 30 seconds. Check server.log.')

### Step 4: Expose Web Portal via Cloudflare Quick Tunnel (No Account Required)
This downloads the official `cloudflared` binary and creates a secure public HTTPS tunnel pointing to your Colab server.

In [ ]:
import os
import re
import subprocess
import time

# Download cloudflared if not present
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Launch tunnel
tunnel_log = open('tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

# Extract public URL
tunnel_url = None
for _ in range(25):
    time.sleep(1)
    with open('tunnel.log', 'r') as f:
        log_text = f.read()
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
    if matches:
        tunnel_url = matches[0]
        break

if tunnel_url:
    print('=' * 65)
    print(f'PUBLIC ACCESS URL: {tunnel_url}')
    print('Open this link in your browser to access the live interface!')
    print('=' * 65)
else:
    print('Could not find Cloudflare URL. Inspect tunnel.log below:')
    !cat tunnel.log

### Step 5: Real-Time Server Telemetry Logs

In [ ]:
# Display recent server output and diagnostic events
!tail -n 40 server.log